# 🚀 Stage 6: Full Modern Detector Fine-Tuning (YOLO11s) on Frozen TarDAL Fusion — Google Colab

This notebook provides the complete, GPU-accelerated experimental pipeline to **investigate whether replacing the legacy YOLOv5su detector with a modern state-of-the-art detector architecture (Ultralytics YOLO11s) improves multi-modal detection accuracy** while keeping the Task-Driven Fusion Generator (**Stage 3 TarDAL Generator**) strictly frozen.

### 🎯 Architectural Thesis & Core Methodology:
- **Multimodal Pipeline**: `RGB + IR → FROZEN Stage-3 TarDAL Generator (stage3_gen_best.pt) → Fused Representation (M3FD_STAGE6_FUSED) → COCO-Pretrained YOLO11s (yolo11s.pt, nc=6) → FULL YOLO11s Fine-Tuning → M3FD Detection`
- **Critical Clarification**: This is **NOT** "replacing the detection head". The YOLOv5su detector was replaced with a COCO-pretrained **YOLO11s detector**, which was **fully fine-tuned** (`freeze=0`, 100% trainable parameters) on the fixed TarDAL fused M3FD representation while the TarDAL generator remained frozen (`generator.eval()`, `requires_grad=False`).
- **Modern Detector Advances (YOLO11s vs. YOLOv5su)**:
  1. **C3k2 Blocks**: Faster multi-scale feature aggregation with reparameterized residual structures.
  2. **C2PSA Attention**: Cross-Stage Partial Spatial Attention modules focusing specifically on salient spatial objects under low-contrast thermal boundaries.
  3. **Refined Decoupled Head**: Advanced Task-Aligned Assigner (TAL) optimizing classification and bounding box regression branches concurrently.
- **Experimental Control & Soundness**:
  1. **Offline Pre-Fusion**: TarDAL generator processes the dataset once to `/content/M3FD_STAGE6_FUSED` (~66.4 ms/img savings during training).
  2. **Strict Parameter Isolation**: Generator is never updated. All YOLO11s weights are updated.
  3. **Controlled Dataset**: Official M3FD 60/20/20 splits (2,520 train, 840 val, 840 test) with 5-column YOLO labels.
  4. **Strict Evaluation Protocol**: Evaluated on official unseen Test split (`imgsz=640`, `conf=0.001`, `iou=0.6`, `batch=16`).

### 📁 Google Drive Path Mapping:
- **Project Code Path**: `/content/drive/MyDrive/FYP/code` (or `/content/drive/MyDrive/fyp/code`)
- **M3FD Dataset Archive**: `/content/drive/MyDrive/FYP/M3FD_Detection.zip`
- **Frozen Generator Checkpoint**: `/content/drive/MyDrive/FYP/code/checkpoints/stage3_gen_best.pt`
- **Stage 6 Outputs**: `/content/drive/MyDrive/FYP/code/checkpoints/stage6/`

### Step 1: Mount Google Drive & Verify GPU Acceleration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU hardware availability
!nvidia-smi

### Step 2: Install Project Dependencies
Ensures `ultralytics >= 8.3.0` is installed for YOLO11 support along with computer vision libraries.

In [ ]:
%pip install -q "ultralytics>=8.3.0" kornia thop tabulate PyYAML tqdm opencv-python matplotlib pandas scipy
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")

### Step 3: Fast & Robust Environment Setup & Dataset Extraction
Resolves repository paths, extracts the `M3FD_Detection.zip` archive to `/content/m3fd/M3FD_Detection` on fast local NVMe SSD storage, and verifies label formatting.

In [ ]:
# @title ⚙️ Step 3: Fast & Robust Environment Setup & Dataset Extraction
import sys
from pathlib import Path

# Dynamic project path resolution
cand_roots = [
    Path('/content/drive/MyDrive/FYP/code'),
    Path('/content/drive/MyDrive/fyp/code'),
    Path('/content/drive/MyDrive/code'),
    Path('/content/code'),
    Path.cwd()
]
CODE_PATH = next((p for p in cand_roots if (p / 'scripts_AG').exists() or (p / 'TarDAL-main').exists()), Path.cwd())

for p in [CODE_PATH, CODE_PATH / 'scripts_AG', CODE_PATH / 'TarDAL-main']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from stage3_colab_setup import setup_stage3_environment
CODE_PATH, ds_root = setup_stage3_environment()
print(f"\n✅ Environment ready at: {CODE_PATH}")
print(f"✅ Dataset active at: {ds_root}")

### Step 4: Pre-Fuse the Entire M3FD Dataset with Frozen TarDAL Generator (Step 16a)
Pre-computes TarDAL task-driven image fusion for all 4,200 pairs (Train: 2,520, Val: 840, Test: 840) to `/content/M3FD_STAGE6_FUSED` on fast SSD:
1. **Tanh Range Remapping**: Applies exact `((fused_y + 1.0) / 2.0).clamp(0.0, 1.0)` formula.
2. **Color Reconstruction**: Inverts YCrCb luminance-chrominance channels back to 3-channel BGR.
3. **Label Sanitation**: Sanitizes label files to strict 5-column format (`class x y w h`) and purges stale Ultralytics `.cache` files.
4. **Split Verification**: Enforces exact dataset split numbers (2,520 train, 840 val, 840 test).

> **Why Pre-Fuse?** Eliminating the generator forward pass during detector training saves ~66.4 ms per image (~46.5 hours over 30 epochs), cutting training time to ~30 minutes on a Tesla T4 GPU.

In [ ]:
# @title ⚡ Step 4: Run High-Speed GPU Pre-Fusion { run: "auto" }
BATCH_SIZE = 16 # @param {type:"integer"}
FORCE_REGENERATE = False # @param {type:"boolean"}

prep_script = str(CODE_PATH / 'scripts_AG' / '16a_prepare_stage6_fused_dataset.py')
prep_cmd = f"python -W ignore {prep_script} --batch_size {BATCH_SIZE}"
if FORCE_REGENERATE:
    prep_cmd += " --force"

print(f"\n>>> {prep_cmd}\n")
get_ipython().system(prep_cmd)

### Step 5: Full Fine-Tuning of YOLO11s on Fixed TarDAL Fused Representation (Step 16b)
Fine-tunes a COCO-pretrained **YOLO11s** (`yolo11s.pt`, 6 classes: `People, Car, Bus, Lamp, Motorcycle, Truck`) on the fused dataset:
- **Hyperparameters**: 30 epochs, batch 16, imgsz 640, AdamW, `lr0=1e-4`, `lrf=0.01`, `cos_lr=True`, `weight_decay=5e-4`, `amp=True`.
- **Parameter Verification**: Strictly asserts `freeze=0` with **100% of YOLO11s parameters trainable** (0 frozen parameters).
- **Sanity Check**: Runs 4-sample synthetic test verifying forward pass, dimension consistency, finite loss, and gradient flow before launching training.
- **Drive Sync**: Checkpoints (`stage6_yolo11s_best.pt`, `stage6_yolo11s_last.pt`), telemetry, training logs, and `results.csv` automatically sync to Google Drive on every epoch completion.
- **Safe Resume**: Resuming training strictly uses Stage 6 checkpoints (`checkpoints/stage6/stage6_yolo11s_last.pt`), preventing accidental contamination from Stage 2/3/4 models.

In [ ]:
# @title 🏋️ Step 5: Train YOLO11s on TarDAL Fused Dataset { run: "auto" }
EPOCHS = 30 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}
LR0 = 1e-4 # @param {type:"number"}
RESUME = False # @param {type:"boolean"}
FORCE_TRAIN = False # @param {type:"boolean"}

train_script = str(CODE_PATH / 'scripts_AG' / '16b_train_stage6_yolo11s.py')
train_cmd = (
    f"python -W ignore {train_script}"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --lr0 {LR0}"
)
if RESUME:
    train_cmd += " --resume"
if FORCE_TRAIN:
    train_cmd += " --force"

print(f"\n>>> {train_cmd}\n")
get_ipython().system(train_cmd)

### Step 6: Benchmark on Unseen Official Test Split (840 Pairs) & True Latency Profile (Step 16c)
Evaluates `stage6_yolo11s_best.pt` on the official M3FD **Test Split** under identical conditions (`imgsz=640`, `batch=16`, `conf=0.001`, `iou=0.6`):
- Reports mAP@50, mAP@50-95, precision, recall, and per-class metrics across all 6 classes.
- **End-to-End Latency Measurement**: Measures the true production deployment latency:
  $$\text{Latency}_{\text{E2E}} = t_{\text{pre}} + t_{\text{TarDAL Generator}} + t_{\text{Color Recon}} + t_{\text{YOLO11s Inference}} + t_{\text{post}}$$
- Saves benchmark results to `/content/drive/MyDrive/FYP/code/checkpoints/stage6/stage6_test_benchmark.json`.

In [ ]:
# @title 🔬 Step 6: Benchmark Stage 6 on Test Split { run: "auto" }
BATCH_SIZE = 16 # @param {type:"integer"}
CONF_THRESH = 0.001 # @param {type:"number"}
IOU_THRESH = 0.60 # @param {type:"number"}

eval_script = str(CODE_PATH / 'scripts_AG' / '16c_eval_stage6_test.py')
eval_cmd = (
    f"python -W ignore {eval_script}"
    f" --batch_size {BATCH_SIZE}"
    f" --conf {CONF_THRESH}"
    f" --iou {IOU_THRESH}"
)

print(f"\n>>> {eval_cmd}\n")
get_ipython().system(eval_cmd)

### Step 7: Qualitative Analysis & 6-Panel Visualizer (Step 16d)
Renders 6-panel qualitative inspection strips across the 5 thesis-critical failure modes:
1. **Visible RGB** (Optical Context)
2. **Thermal IR** (Thermal Radiation)
3. **TarDAL Fused Image** (Fused Representation)
4. **Legacy YOLOv5su Detections** (`stage3_best.pt`)
5. **Modern YOLO11s Detections** (`stage6_yolo11s_best.pt`)
6. **Ground Truth Annotations**

> **Thesis Case Studies Targeted**:
> - Case 1: YOLO11s detects a target missed by YOLOv5su.
> - Case 2: Legacy YOLOv5su detects an object missed by YOLO11s.
> - Case 3: YOLO11s produces tighter bounding box localization on thermal edges.
> - Case 4: YOLO11s correctly suppresses a false positive hallucinated by YOLOv5su.
> - Case 5: Distant small objects (`People`, `Lamp`) under heavy sensor degradation.

In [ ]:
# @title 🎨 Step 7: Generate 6-Panel Qualitative Visualizations { run: "auto" }
NUM_SAMPLES = 10 # @param {type:"integer"}
CONF_VIS = 0.25 # @param {type:"number"}
IOU_VIS = 0.50 # @param {type:"number"}

vis_script = str(CODE_PATH / 'scripts_AG' / '16d_visualize_stage6_results.py')
vis_cmd = (
    f"python -W ignore {vis_script}"
    f" --num_samples {NUM_SAMPLES}"
    f" --conf {CONF_VIS}"
    f" --iou {IOU_VIS}"
)

print(f"\n>>> {vis_cmd}\n")
get_ipython().system(vis_cmd)

### Step 8: Master Comparative Benchmark & FYP Hypothesis Evaluation (Step 16e)
Synthesizes the full benchmark across all five architectures evaluated throughout the FYP:
1. **Direct Optical RGB** (YOLOv5su Unimodal)
2. **Direct Thermal IR** (YOLOv5su Unimodal)
3. **Stage 3 TarDAL + YOLOv5su** (Feature-Level Task-Driven Fusion)
4. **Stage 5 Late Fusion WBF** (Decision-Level Fusion)
5. **Stage 6 TarDAL + YOLO11s** (Modern Detector Fine-Tuned on Frozen Fusion)

Directly evaluates and answers the **9 Thesis Hypothesis Questions** (Section 40):
- Does YOLO11s outperform YOLOv5su on TarDAL fused images?
- Which classes experience the largest gains?
- Is the improvement uniform or class-selective?
- Does the modern detector improve small object detection (`People`, `Lamp`)?
- What is the latency and throughput trade-off?
- How does modern detector fusion compare to decision-level late fusion?
- Does modern detector fine-tuning alter the architectural recommendation for the FYP?
- Generates Pareto latency vs. accuracy frontier plots and exports reports to Google Drive.

In [ ]:
# @title 🏆 Step 8: Generate Master Comparative Benchmark & FYP Defense Report { run: "auto" }
comp_script = str(CODE_PATH / 'scripts_AG' / '16e_compare_stage6_results.py')
comp_cmd = f"python -W ignore {comp_script}"

print(f"\n>>> {comp_cmd}\n")
get_ipython().system(comp_cmd)